### 1. 消息概述

在 LangChain 中，消息是模型的基本上下文单元。它们代表模型的输入和输出，携带与 LLM 交互时表示对话状态所需的内容和元数据。

消息是包含以下内容的对象：
- 角色 —— 标识消息类型（例如 system，user，assistant）
- 内容 —— 指消息的实际内容（例如文本、图像、音频、文档等）。
- 元数据 —— 可选字段，例如响应信息、消息 ID 和令牌使用情况

LangChain 提供了一种适用于所有模型提供程序的标准消息类型，确保无论调用哪个模型，行为都保持一致。

为什么要区分 role？因为模型是"无状态"的——它不会记住上一轮。所以我们每次都要把整段对话（一串带 role 的消息）一起传给模型，模型靠 role 判断"哪些是背景、哪些是问题、哪些是我之前说过的话"。

### 2. 四种消息类型

In [6]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

# ⭐ 四种消息，各司其职
system = SystemMessage(content="你是一个耐心的 Python 老师")
human = HumanMessage(content="什么是装饰器？")
ai = AIMessage(content="装饰器是一个接收函数、返回新函数的函数……")
tool = ToolMessage(content="查询结果：25°C", tool_call_id="call_001")

# 观察每种消息的类型和 content
for msg in [system, human, ai, tool]:
    print(type(msg).__name__, "=>", msg.content)

SystemMessage => 你是一个耐心的 Python 老师
HumanMessage => 什么是装饰器？
AIMessage => 装饰器是一个接收函数、返回新函数的函数……
ToolMessage => 查询结果：25°C


| 消息类型 | role | 谁说的 | 用途 |
|---------|------|--------|------|
| SystemMessage | system | 开发者 | 定人设、定规则（语气、格式、角色）|
| HumanMessage | user | 用户 | 提问、下指令 |
| AIMessage | assistant | 模型 | 模型的回答 |
| ToolMessage | tool | 工具 | 工具执行结果（Agent 阶段用）|

工具消息

In [8]:
from langchain.chat_models import init_chat_model
model = init_chat_model(model="deepseek-flash")

# 定义工具并调用
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    print('tool调用：get_weather')
    return f'今天天气真好，在 {location} 这个魅力的城市！'

model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("长沙天气怎么样?")

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")  # Tool: get_weather 工具名（即函数名）
    print(f"Args: {tool_call['args']}")  # Args: {'location': '长沙'} 传给工具的参数
    print(f"ID: {tool_call['id']}")  # ID: call_e68e9d4bd0e64d8a8c54ba 唯一标识符，用于后续匹配工具执行结果
print('-' * 100)
print(response)


Tool: get_weather
Args: {'location': '长沙'}
ID: call_00_CO09oI1LX3z0I0gf8iIc2761
----------------------------------------------------------------------------------------------------
content='' additional_kwargs={'refusal': None, 'reasoning_content': 'The user asks about the weather in Changsha. I should call the get_weather tool.'} response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 293, 'total_tokens': 351, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 19, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 165}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'bbc46cdb-b61b-40c1-9bf5-34cadc3d6e49', 'finish_reason': 'to

两种写法等价：字典格式 ↔ Message 对象（推荐用对象，可读性更好、功能更全）

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

model = init_chat_model(model="deepseek-flash")

# 写法一：字典格式
conversation_dict = [
    {"role": "system", "content": "你是翻译助手"},
    {"role": "user", "content": "翻译：我喜欢编程"},
]

# 写法二：Message 对象（推荐）
conversation_obj = [
    SystemMessage(content="你是翻译助手"),
    HumanMessage(content="翻译：我喜欢编程"),
]

# 两种写法效果完全一样
print("字典格式：", model.invoke(conversation_dict).content)
print("对象格式：", model.invoke(conversation_obj).content)

### 3. 消息的内部结构

一条 AIMessage 不止有 content，还携带很多元信息。用 model_dump_json() 可以看到完整字段。

In [1]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="deepseek-flash")
response = model.invoke("你好")

# ⭐ 查看 AIMessage 的完整字段
print(response.model_dump_json(indent=2))

{
  "content": "你好！很高兴见到你。有什么我可以帮你的吗？😊",
  "additional_kwargs": {
    "refusal": null,
    "reasoning_content": "The user just said \"你好\" (hello) in Chinese. I should respond in Chinese with a friendly greeting and offer to help."
  },
  "response_metadata": {
    "token_usage": {
      "completion_tokens": 42,
      "prompt_tokens": 31,
      "total_tokens": 73,
      "completion_tokens_details": {
        "accepted_prediction_tokens": null,
        "audio_tokens": null,
        "reasoning_tokens": 27,
        "rejected_prediction_tokens": null,
        "text_tokens": null
      },
      "prompt_tokens_details": {
        "audio_tokens": null,
        "cache_write_tokens": null,
        "cached_tokens": 0,
        "image_tokens": null,
        "text_tokens": null
      },
      "prompt_cache_hit_tokens": 0,
      "prompt_cache_miss_tokens": 31
    },
    "model_provider": "deepseek",
    "model_name": "deepseek-flash",
    "system_fingerprint": "aeb56401ca74e127821c4f9126dcb669",
   

```
AIMessage
├── content              最终文本
├── additional_kwargs    附加信息（如推理内容 reasoning_content）
├── response_metadata    模型原始元数据（token 用量、模型名）
├── id                   消息唯一 ID
├── tool_calls           工具调用（Agent 阶段用）
└── usage_metadata       标准化 token 用量
```

In [2]:
# ⭐ 逐字段访问
print("正文：", response.content)
print("推理内容：", response.additional_kwargs.get("reasoning_content"))
print("token 用量：", response.usage_metadata)
print("模型元数据：", response.response_metadata)

正文： 你好！很高兴见到你。有什么我可以帮你的吗？😊
推理内容： The user just said "你好" (hello) in Chinese. I should respond in Chinese with a friendly greeting and offer to help.
token 用量： {'input_tokens': 31, 'output_tokens': 42, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 27}}
模型元数据： {'token_usage': {'completion_tokens': 42, 'prompt_tokens': 31, 'total_tokens': 73, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 27, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '9c85094f-a0d5-4fa2-9ede-ccbe3e102e16', 'finish_reason': 'stop', 'logprobs': None}


### 4. 构建完整的多轮对话

把多轮消息按顺序放进列表，就是一次完整对话。模型会根据 role 理解上下文。

In [3]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

model = init_chat_model(model="deepseek-flash")

# ⭐ 多轮对话 = 按顺序的消息列表
conversation = [
    SystemMessage(content="你是一个记性很好的助手"),
    HumanMessage(content="我叫小明"),
    AIMessage(content="你好小明！"),
    HumanMessage(content="我叫什么名字？"),
]

response = model.invoke(conversation)
print(response.content)  # 模型能"记得"前面说过的名字

你叫小明。
